In [3]:
import math
import torch
from torch import nn
from d2l import torch as d2l

遮蔽softmax操作

In [4]:
# 序列长度不一致时，需要对序列进行填充，但填充算softmax无意义，就要屏蔽掉填充
def masked_softmax(X, valid_lens):
    '''通过最后一个轴上遮蔽元素来执行 softmax 操作'''
    # 没有有效长度，正常做softmax
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        # 保存shape，后面reshape用
        shape = X.shape
        # 如果维度为1
        if valid_lens.dim() == 1:
            # 因为X要reshape成将batch和query合并，(2,4,6)->(8,6) 所以要将valid_lens[i]重复多query次,以对应reshape后每一行的有效key的个数
            valid_lens = torch.repeat_interleave(valid_lens, shape[1])  # shape[1] 是 query 数量（3）
        # 如果维度不为1，可以理解为给出了每一行的有效key的个数，直接拍平即可
        else:
            valid_lens = valid_lens.reshape(-1) 
        # 将每个独立的batch，合并成多行，序列长度不足的用value来填充
        X = d2l.sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6) #shape[-1]每个batch里的key的个数
        # 最后reshape回原来的形状
    return nn.functional.softmax(X.reshape(shape), dim=-1)

In [5]:
masked_softmax(torch.rand(2, 2, 4), torch.tensor([2, 3]))

tensor([[[0.6859, 0.3141, 0.0000, 0.0000],
         [0.3458, 0.6542, 0.0000, 0.0000]],

        [[0.2539, 0.2879, 0.4582, 0.0000],
         [0.2884, 0.4371, 0.2745, 0.0000]]])

加性注意力:   score(q, k) = v^T · tanh(W_q·q + W_k·k)   ← q 和 k 先各自投影再加起来，过 MLP

In [ ]:
class AdditiveAttention(nn.Block):
    """加性注意力"""
    def __init__(self, num_hiddens, dropout, **kwargs):
        super(AdditiveAttention, self).__init__(**kwargs)
        # 使用'flatten=False'只转换最后一个轴，以便其他轴的形状保持不变
        self.W_k = nn.Dense(num_hiddens, use_bias=False, flatten=False)
        self.W_q = nn.Dense(num_hiddens, use_bias=False, flatten=False)
        self.w_v = nn.Dense(1, use_bias=False, flatten=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens):
        queries, keys = self.W_q(queries), self.W_k(keys)
        features = queries.unsqueeze(2) + keys.unsqueeze(1)
        features = torch.tanh(features)
        # self.w_v仅有一个输出，因此从形状中移除最后那个维度。
        scores = self.w_v(features).squeeze(-1)
        self.attention_weights = masked_softmax(scores, valid_lens)
        # values的形状：(batch_size，“键－值”对的个数，值的维度)
        return torch.bmm(self.dropout(self.attention_weights), values)

squeeze(-1) 是 unsqueeze 的反操作——删除尺寸为 1 的维度。-1 指最后一个维度。
原始 (3, 4):           

 [a b c d]            ← 3 行 4 列  
 [e f g h]  
 [i j k l]

unsqueeze(0):         变成 1 个"3×4 的板子"  (1, 3, 4)  
unsqueeze(1):         每行变成 1×4 的一维数组  (3, 1, 4)    
unsqueeze(2):         每个元素变成 1 维向量    (3, 4, 1)  

## 缩放点积注意力

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

问题出在点积的方差。假设 $q$ 和 $k$ 的每个分量是独立、均值 0、方差 1 的随机变量：

$$q \cdot k = \sum_{i=1}^{d_k} q_i k_i$$

$d_k$ 项相加，每项方差是 1，点积结果的方差 = $d_k$。

$d_k$ 小的时候没事。$d_k$ 大（比如 128、512），点积的绝对值可能很大，softmax 输入值过大 → 梯度趋近 0 → softmax 进入饱和区，梯度消失，训不动。

除以 $\sqrt{d_k}$ 把方差压回 1：

$$\text{Var}\left(\frac{q \cdot k}{\sqrt{d_k}}\right) = \frac{d_k}{d_k} = 1$$

$d_k$就是 q（或 k）这个向量里有几个数。

In [ ]:
class DotProductAttention(nn.Block):
    """缩放点积注意力"""
    def __init__(self, dropout, **kwargs):
        super(DotProductAttention, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)

    # queries的形状：(batch_size，查询的个数，d)
    # keys的形状：(batch_size，“键－值”对的个数，d)
    # values的形状：(batch_size，“键－值”对的个数，值的维度)
    # valid_lens的形状:(batch_size，)或者(batch_size，查询的个数)
    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]  # q这个向量里有多少个数
        scores = torch.bmm(queries, keys.transpose(1,2)) / math.sqrt(d)  # 注意力分数
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)  